# XLS-R full fine-tuning
based on https://huggingface.co/blog/fine-tune-xlsr-wav2vec2


In [1]:
%pip install -U pip
%pip install --no-cache-dir 'transformers==4.57.1' accelerate 'datasets[audio]' evaluate jiwer safetensors huggingface_hub tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 73.0 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 296.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 873.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 1.3 GB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 1.1 GB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 686.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 309.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 385.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 1.2 GB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 319.1 MB/s  0:00:00
  Attempting uninstall: huggin

In [2]:
import os
import re
import json
import gc
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Union

import numpy as np
import pandas as pd
import torch
import jiwer

from tqdm.auto import tqdm
from datasets import load_dataset, Audio
from huggingface_hub import notebook_login, HfFolder, create_repo

from transformers import (Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor,
    Wav2Vec2ForCTC, TrainingArguments, Trainer, EarlyStoppingCallback, set_seed)

In [3]:
notebook_login()

In [5]:
hf_token = HfFolder.get_token()

if hf_token is None:
    raise ValueError('HF token was not found. Run notebook_login() first.')

In [6]:
def create_xlsr_model(model_id):
    model = Wav2Vec2ForCTC.from_pretrained(
        model_id,
        attention_dropout=0.0,
        hidden_dropout=0.0,
        feat_proj_dropout=0.0,
        mask_time_prob=0.05,
        layerdrop=0.0,
        ctc_loss_reduction='mean',
        pad_token_id=processor.tokenizer.pad_token_id,
        vocab_size=len(processor.tokenizer),
        ignore_mismatched_sizes=True
    )

    model.freeze_feature_encoder()

    return model


def extract_all_chars(batch):
    all_text = ' '.join(batch['sentence'])
    vocab = sorted(list(set(all_text)))
    return {'vocab': [vocab], 'all_text': [all_text]}


def prepare_dataset(batch):
    audio = batch['audio']

    batch['input_values'] = processor(audio['array'], sampling_rate=audio['sampling_rate']).input_values[0]
    batch['input_length'] = len(batch['input_values'])
    batch['labels'] = processor(text=batch['sentence']).input_ids

    return batch


@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{'input_values': feature['input_values']} for feature in features]
        label_features = [{'input_ids': feature['labels']} for feature in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors='pt')
        labels_batch = self.processor.pad(labels=label_features, padding=self.padding, return_tensors='pt')

        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch['labels'] = labels

        return batch


def normalize_spaces(text):
    if text is None:
        return ''

    text = str(text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)

    label_str = processor.batch_decode(label_ids, group_tokens=False)

    pred_str = [normalize_spaces(text) for text in pred_str]
    label_str = [normalize_spaces(text) for text in label_str]

    wer = jiwer.wer(label_str, pred_str)
    cer = jiwer.cer(label_str, pred_str)

    return {'wer': wer, 'cer': cer}


def print_trainable_parameters(model):
    trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
    all_params = sum(param.numel() for param in model.parameters())

    print(f'Trainable params: {trainable_params:,}')
    print(f'All params: {all_params:,}')
    print(f'Trainable share: {100 * trainable_params / all_params:.4f}%')


def get_best_dev_metrics(trainer, log_history_df):
    best_checkpoint = trainer.state.best_model_checkpoint
    best_dev_cer = trainer.state.best_metric

    best_step = None
    best_dev_loss = None
    best_dev_wer = None

    if best_checkpoint is not None:
        match = re.search(r'checkpoint-(\d+)', best_checkpoint)

        if match is not None:
            best_step = int(match.group(1))

            best_eval_rows = log_history_df[
                (log_history_df['step'] == best_step) &
                (log_history_df['eval_cer'].notna())
            ]

            if len(best_eval_rows) > 0:
                best_eval_row = best_eval_rows.iloc[0]
                best_dev_loss = best_eval_row['eval_loss']
                best_dev_wer = best_eval_row['eval_wer']
                best_dev_cer = best_eval_row['eval_cer']

    return {
        'best_dev_checkpoint': best_checkpoint,
        'best_dev_step': best_step,
        'best_dev_loss': best_dev_loss,
        'best_dev_WER': best_dev_wer,
        'best_dev_CER': best_dev_cer
    }


def prepare_resource_test_dataset(resource):
    test_dataset_raw_resource = dataset_dict['test'].filter(lambda example: example['resource'] == resource)

    if len(test_dataset_raw_resource) == 0:
        raise ValueError(f'No test examples found for resource: {resource}')

    test_dataset_resource = test_dataset_raw_resource.map(prepare_dataset,
        remove_columns=test_dataset_raw_resource.column_names, load_from_cache_file=False)

    decoded_labels = processor.batch_decode(test_dataset_resource['labels'], group_tokens=False)
    decoded_labels = [normalize_spaces(text) for text in decoded_labels]

    references = [normalize_spaces(text) for text in test_dataset_raw_resource['sentence']]
    mismatches = [(i, ref, label) for i, (ref, label) in enumerate(zip(references, decoded_labels)) if ref != label]

    print(f'{resource} mismatches before predict:', len(mismatches))

    if len(mismatches) > 0:
        print(mismatches[:5])
        raise ValueError(f'{resource}: raw references and prepared labels do not match')

    return test_dataset_raw_resource, test_dataset_resource


def predict_dataset_in_order(model, prepared_dataset, raw_dataset, data_collator, processor, batch_size=8):
    device = next(model.parameters()).device
    model.eval()

    rows = []

    for start in tqdm(range(0, len(prepared_dataset), batch_size)):
        end = min(start + batch_size, len(prepared_dataset))

        features = [prepared_dataset[i] for i in range(start, end)]

        batch = data_collator(features)

        input_batch = {key: value.to(device) for key, value in batch.items() if key != 'labels'}

        with torch.no_grad():
            logits = model(**input_batch).logits

        pred_ids = torch.argmax(logits, dim=-1)

        pred_str = processor.batch_decode(pred_ids)
        pred_str = [normalize_spaces(text) for text in pred_str]

        for local_i, prediction in enumerate(pred_str):
            raw_i = start + local_i
            raw_example = raw_dataset[raw_i]

            rows.append({
                'resource': raw_example['resource'],
                'path': raw_example['path'],
                'reference': normalize_spaces(raw_example['sentence']),
                'prediction': prediction
            })

    return pd.DataFrame(rows)


def evaluate_resource_test(resource, experiment_name, model, batch_size=8):
    test_dataset_raw_resource, test_dataset_resource = prepare_resource_test_dataset(resource)

    results_df = predict_dataset_in_order(model=model, prepared_dataset=test_dataset_resource,
        raw_dataset=test_dataset_raw_resource, data_collator=data_collator,
        processor=processor, batch_size=batch_size)

    wer = jiwer.wer(results_df['reference'].tolist(), results_df['prediction'].tolist())

    cer = jiwer.cer(results_df['reference'].tolist(), results_df['prediction'].tolist())

    predictions_path = f'{experiment_name}_{resource}_test_predictions.csv'

    results_df.to_csv(predictions_path, index=False, encoding='utf-8-sig')

    print(f'{resource}_test WER:', wer)
    print(f'{resource}_test CER:', cer)

    row = {
        'model': model_label,
        'training': experiment_name,
        'subset': f'{resource}_test',
        'WER': wer,
        'CER': cer,
        'n_files': len(results_df),
        'predictions_file': predictions_path
    }

    return row, results_df

In [7]:
dataset_repo_id = 'tadgeis/chukchi-asr-data-private'

dataset_dict = load_dataset(dataset_repo_id, token=hf_token)
dataset_dict = dataset_dict.cast_column('audio', Audio(sampling_rate=16_000))

dataset_dict

README.md:   0%|          | 0.00/422 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/411M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/410M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/145M [00:00<?, ?B/s]

data/dev-00000-of-00001.parquet:   0%|          | 0.00/60.0M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating dev split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
        num_rows: 2714
    })
    test: Dataset({
        features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
        num_rows: 429
    })
    dev: Dataset({
        features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
        num_rows: 140
    })
})

In [8]:
train_chars = set(' '.join(dataset_dict['train']['sentence']))
dev_chars = set(' '.join(dataset_dict['dev']['sentence']))
test_chars = set(' '.join(dataset_dict['test']['sentence']))

print('Chars in dev but not train:', dev_chars - train_chars)
print('Chars in test but not train:', test_chars - train_chars)

print('Train chars:', sorted(train_chars))

Chars in dev but not train: set()
Chars in test but not train: set()
Train chars: [' ', "'", 'а', 'б', 'в', 'г', 'д', 'е', 'ж', 'з', 'и', 'й', 'к', 'л', 'м', 'н', 'о', 'п', 'р', 'с', 'т', 'у', 'ф', 'х', 'ц', 'ч', 'ш', 'щ', 'ъ', 'ы', 'ь', 'э', 'ю', 'я', 'ё', 'ӄ', 'ӈ', 'ԓ']


In [9]:
vocab_source_dataset = dataset_dict['train']

vocab_train = vocab_source_dataset.map(extract_all_chars, batched=True, batch_size=-1,
    keep_in_memory=True, remove_columns=vocab_source_dataset.column_names)

vocab_list = vocab_train['vocab'][0]
vocab_dict = {char: idx for idx, char in enumerate(vocab_list)}

vocab_dict['|'] = vocab_dict[' ']
del vocab_dict[' ']

vocab_dict['[UNK]'] = len(vocab_dict)
vocab_dict['[PAD]'] = len(vocab_dict)

Map:   0%|          | 0/2714 [00:00<?, ? examples/s]

In [10]:
with open('vocab.json', 'w', encoding='utf-8') as vocab_file:
    json.dump(vocab_dict, vocab_file, ensure_ascii=False, indent=2)

In [11]:
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained('./', unk_token='[UNK]', pad_token='[PAD]', word_delimiter_token='|')

In [12]:
sample_text = dataset_dict['train'][0]['sentence']

encoded = tokenizer(sample_text).input_ids
decoded = tokenizer.decode(encoded, group_tokens=False)

print('Original:', sample_text)
print('Encoded:', encoded)
print('Decoded:', decoded)

Original: ымыԓьо кэлит тыӈивылӄылти ӄликкин январьтагнэты
Encoded: [29, 14, 29, 37, 30, 16, 0, 12, 31, 13, 10, 20, 0, 20, 29, 36, 10, 4, 29, 13, 35, 29, 13, 20, 10, 0, 35, 13, 10, 12, 12, 10, 15, 0, 33, 15, 4, 2, 18, 30, 20, 2, 5, 15, 31, 20, 29]
Decoded: ымыԓьо кэлит тыӈивылӄылти ӄликкин январьтагнэты


In [13]:
feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16_000,
    padding_value=0.0, do_normalize=True, return_attention_mask=True)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

In [14]:
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

# staged source-eval fine-tuning
## Stage 1
    fine-tune XLS-R on source-domain data: bible + radio.  
    Validation for Stage 1 is also source-domain: bible_test + radio_test (!!!).

In [16]:
experiment_name = 'xlsr_300m_staged_bible_radio_to_chuklang_source_eval'
model_id = 'facebook/wav2vec2-xls-r-300m'

stage1_repo_id = 'tadgeis/xls-r-300m-ckt-staged-bible-radio-to-chuklang-source-eval-stage1'
model_repo_id = 'tadgeis/xls-r-300m-ckt-staged-bible-radio-to-chuklang-source-eval-stage2'

model_label = 'XLS-R 300M CTC fine-tuning'

source_resources = ['bible', 'radio']
target_resource = 'chuklang'

# Only chuklang_test is a final held-out test.
# radio_test and bible_test were used for Stage 1 source-domain eval,
# so they must not be reported as final test scores.
resources_to_evaluate = ['chuklang', 'radio', 'bible']

In [17]:
for repo_id in [stage1_repo_id, model_repo_id]:
    create_repo(
        repo_id=repo_id,
        repo_type='model',
        private=False,
        exist_ok=True,
        token=hf_token
    )

    processor.push_to_hub(
        repo_id,
        private=False,
        token=hf_token
    )


create_repo(repo_id=model_repo_id, repo_type='model', private=False, exist_ok=True, token=hf_token)
processor.push_to_hub(model_repo_id, private=False, token=hf_token)

README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/tadgeis/xls-r-300m-ckt-staged-bible-radio-to-chuklang-source-eval-stage2/commit/10b5fe6ad2a998e95bdcfb3796c4e664dbb89dda', commit_message='Upload processor', commit_description='', oid='10b5fe6ad2a998e95bdcfb3796c4e664dbb89dda', pr_url=None, repo_url=RepoUrl('https://huggingface.co/tadgeis/xls-r-300m-ckt-staged-bible-radio-to-chuklang-source-eval-stage2', endpoint='https://huggingface.co', repo_type='model', repo_id='tadgeis/xls-r-300m-ckt-staged-bible-radio-to-chuklang-source-eval-stage2'), pr_revision=None, pr_num=None)

In [18]:
SEED = 42
set_seed(SEED)

stage1_train_dataset_raw = dataset_dict['train'].filter(
    lambda example: example['resource'] in source_resources
)

stage1_eval_dataset_raw = dataset_dict['test'].filter(
    lambda example: example['resource'] in source_resources
)

stage2_train_dataset_raw = dataset_dict['train'].filter(
    lambda example: example['resource'] == target_resource
)

stage2_eval_dataset_raw = dataset_dict['dev'].filter(
    lambda example: example['resource'] == target_resource
)

stage1_train_dataset_raw = stage1_train_dataset_raw.shuffle(seed=SEED)
stage2_train_dataset_raw = stage2_train_dataset_raw.shuffle(seed=SEED)

print('Stage 1 train raw:', stage1_train_dataset_raw)
print('Stage 1 eval raw:', stage1_eval_dataset_raw)
print('Stage 2 train raw:', stage2_train_dataset_raw)
print('Stage 2 eval raw:', stage2_eval_dataset_raw)

Filter:   0%|          | 0/2714 [00:00<?, ? examples/s]

Filter:   0%|          | 0/429 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2714 [00:00<?, ? examples/s]

Filter:   0%|          | 0/140 [00:00<?, ? examples/s]

Stage 1 train raw: Dataset({
    features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
    num_rows: 2055
})
Stage 1 eval raw: Dataset({
    features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
    num_rows: 229
})
Stage 2 train raw: Dataset({
    features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
    num_rows: 659
})
Stage 2 eval raw: Dataset({
    features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
    num_rows: 140
})


In [19]:
stage1_train_dataset = stage1_train_dataset_raw.map(
    prepare_dataset,
    remove_columns=stage1_train_dataset_raw.column_names,
    load_from_cache_file=False
)

stage1_eval_dataset = stage1_eval_dataset_raw.map(
    prepare_dataset,
    remove_columns=stage1_eval_dataset_raw.column_names,
    load_from_cache_file=False
)

stage2_train_dataset = stage2_train_dataset_raw.map(
    prepare_dataset,
    remove_columns=stage2_train_dataset_raw.column_names,
    load_from_cache_file=False
)

stage2_eval_dataset = stage2_eval_dataset_raw.map(
    prepare_dataset,
    remove_columns=stage2_eval_dataset_raw.column_names,
    load_from_cache_file=False
)

print('Stage 1 prepared train:', stage1_train_dataset)
print('Stage 1 prepared eval:', stage1_eval_dataset)
print('Stage 2 prepared train:', stage2_train_dataset)
print('Stage 2 prepared eval:', stage2_eval_dataset)

Map:   0%|          | 0/2055 [00:00<?, ? examples/s]

Map:   0%|          | 0/229 [00:00<?, ? examples/s]

Map:   0%|          | 0/659 [00:00<?, ? examples/s]

Map:   0%|          | 0/140 [00:00<?, ? examples/s]

Stage 1 prepared train: Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 2055
})
Stage 1 prepared eval: Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 229
})
Stage 2 prepared train: Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 659
})
Stage 2 prepared eval: Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 140
})


In [20]:
model = create_xlsr_model(model_id)

print_trainable_parameters(model)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-xls-r-300m and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable params: 311,271,594
All params: 315,481,770
Trainable share: 98.6655%


In [21]:
# # test on the longest audio files

# def test_longest_batch(model, train_dataset, data_collator, batch_size=4):
#     if not torch.cuda.is_available():
#         raise RuntimeError('CUDA is not available.')

#     model.to('cuda')
#     model.train()
#     model.zero_grad(set_to_none=True)

#     longest_indices = np.argsort(-np.array(train_dataset['input_length']))[:batch_size]

#     print('Testing indices:', longest_indices.tolist())
#     print('Lengths in seconds:',
#         [round(train_dataset[int(i)]['input_length'] / 16000, 2) for i in longest_indices]
#     )

#     features = [train_dataset[int(i)] for i in longest_indices]

#     batch = data_collator(features)

#     batch = {key: value.to('cuda') for key, value in batch.items()}

#     try:
#         with torch.autocast(device_type='cuda', dtype=torch.float16):
#             outputs = model(**batch)
#             loss = outputs.loss

#         loss.backward()

#         print('Loss:', loss.item())
#         print('Longest-batch train forward/backward: PASSED')

#     finally:
#         model.zero_grad(set_to_none=True)

#         del batch
#         del features
#         del outputs
#         del loss

#         gc.collect()
#         torch.cuda.empty_cache()

#         !nvidia-smi


# ## test batch size
# test_longest_batch(
#     model=model,
#     train_dataset=stage1_train_dataset,
#     data_collator=data_collator,
#     batch_size=16
# )

# test_longest_batch(
#     model=model,
#     train_dataset=stage2_train_dataset,
#     data_collator=data_collator,
#     batch_size=16
# )

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

In [43]:
stage1_training_args = TrainingArguments(
    output_dir=stage1_repo_id.split('/')[-1],

    group_by_length=True,

    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=4,

    eval_strategy='steps',
    save_strategy='steps',

    num_train_epochs=6,

    gradient_checkpointing=False,
    fp16=torch.cuda.is_available(),

    save_steps=100,
    eval_steps=100,
    logging_steps=50,

    learning_rate=3e-4,
    lr_scheduler_type='constant',
    warmup_steps=0,

    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,

    push_to_hub=True,
    hub_model_id=stage1_repo_id,
    hub_private_repo=False,
    hub_token=hf_token,
    hub_strategy='checkpoint',
    hub_always_push=True,

    report_to='none',
    disable_tqdm=False,

    seed=SEED,
    data_seed=SEED
)


stage2_training_args = TrainingArguments(
    output_dir=model_repo_id.split('/')[-1],

    group_by_length=True,

    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=4,

    eval_strategy='steps',
    save_strategy='steps',

    num_train_epochs=12,

    gradient_checkpointing=False,
    fp16=torch.cuda.is_available(),

    save_steps=50,
    eval_steps=50,
    logging_steps=50,

    learning_rate=1e-5,
    warmup_steps=0,

    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,

    push_to_hub=True,
    hub_model_id=model_repo_id,
    hub_private_repo=False,
    hub_token=hf_token,
    hub_strategy='checkpoint',
    hub_always_push=True,

    report_to='none',
    disable_tqdm=False,

    seed=SEED,
    data_seed=SEED
)

  .../checkpoint-300/scaler.pt: 100%|##########| 1.38kB / 1.38kB            

In [34]:
stage1_trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=stage1_training_args,
    compute_metrics=compute_metrics,
    train_dataset=stage1_train_dataset,
    eval_dataset=stage1_eval_dataset,
    processing_class=processor.feature_extractor,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=5,
            early_stopping_threshold=0.001
        )
    ]
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-stage2/training_args.bin: 100%|##########| 6.03kB / 6.03kB            

  ...-stage2/model.safetensors:   0%|          | 3.86MB / 1.26GB            

In [24]:
!rm -rf xlsr-1b-ckt-chuklang-only/checkpoint-*
!rm -rf xlsr-1b-ckt-staged-bible-radio-to-chuklang-source-eval-stage1/checkpoint-*
!rm -rf xlsr-1b-ckt-staged-bible-radio-to-chuklang-source-eval-stage2/checkpoint-*

In [25]:
!df -h
!du -h --max-depth=1 . | sort -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         200G  6.3G  194G   4% /
tmpfs            64M     0   64M   0% /dev
shm             157G  4.0K  157G   1% /dev/shm
/dev/md0        7.0T  1.9T  5.2T  27% /etc/hosts
tmpfs           126G  1.6M  126G   1% /run/nvidia-persistenced/socket
/dev/root       124G   11G  114G   9% /usr/bin/nvidia-smi
tmpfs           315G     0  315G   0% /proc/acpi
tmpfs           315G     0  315G   0% /proc/scsi
tmpfs           315G     0  315G   0% /sys/firmware
du: cannot read directory './proc/929/task/929/fdinfo': Permission denied
du: cannot read directory './proc/929/map_files': Permission denied
du: cannot read directory './proc/929/fdinfo': Permission denied
du: cannot read directory './proc/951/task/951/fdinfo': Permission denied
du: cannot read directory './proc/951/map_files': Permission denied
du: cannot read directory './proc/951/fdinfo': Permission denied
du: cannot read directory './proc/1041/task/1041/fdinfo': Permission denied
du: 

In [26]:
stage1_trainer.train()

Step,Training Loss,Validation Loss,Wer,Cer
100,3.071100,3.075181,1.000000,1.000000
200,2.983200,2.919065,1.000000,0.980845
300,2.363700,2.048807,1.000000,0.643210
400,1.509800,1.218947,0.999590,0.375045
500,1.065900,0.849741,0.882377,0.244874
600,0.826300,0.649351,0.736475,0.170594
700,0.703600,0.553709,0.691803,0.151079
800,0.709400,0.514474,0.626639,0.136061
900,0.691100,0.500072,0.604098,0.128597
1000,0.620300,0.446566,0.576230,0.121718


No files have been modified since last commit. Skipping to prevent empty commit.


TrainOutput(global_step=3084, training_loss=0.8253405211655237, metrics={'train_runtime': 1242.0445, 'train_samples_per_second': 9.927, 'train_steps_per_second': 2.483, 'total_flos': 3.4229572868739323e+18, 'train_loss': 0.8253405211655237, 'epoch': 6.0})

In [28]:
stage1_log_history_df = pd.DataFrame(stage1_trainer.state.log_history)

stage1_log_history_df.to_csv(
    f'{experiment_name}_stage1_log_history.csv',
    index=False,
    encoding='utf-8-sig'
)

stage1_best_metrics = get_best_dev_metrics(
    stage1_trainer,
    stage1_log_history_df
)

print(stage1_best_metrics)

{'best_dev_checkpoint': 'xls-r-300m-ckt-staged-bible-radio-to-chuklang-source-eval-stage1/checkpoint-2600', 'best_dev_step': 2600, 'best_dev_loss': np.float64(0.358144074678421), 'best_dev_WER': np.float64(0.4344262295081967), 'best_dev_CER': np.float64(0.08444244604316546)}


In [29]:
stage1_trainer.save_model(stage1_training_args.output_dir)
stage1_trainer.push_to_hub()

processor.push_to_hub(
    stage1_repo_id,
    private=False,
    token=hf_token
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/tadgeis/xls-r-300m-ckt-staged-bible-radio-to-chuklang-source-eval-stage1/commit/26abb65fe488e04d152f4bf77ca1476681a19ae9', commit_message='Upload processor', commit_description='', oid='26abb65fe488e04d152f4bf77ca1476681a19ae9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/tadgeis/xls-r-300m-ckt-staged-bible-radio-to-chuklang-source-eval-stage1', endpoint='https://huggingface.co', repo_type='model', repo_id='tadgeis/xls-r-300m-ckt-staged-bible-radio-to-chuklang-source-eval-stage1'), pr_revision=None, pr_num=None)

## Stage 2:
    continue from best Stage 1 checkpoint
    train: chuklang_train
    eval:  chuklang_dev

In [44]:
stage2_trainer = Trainer(
    model=stage1_trainer.model,
    data_collator=data_collator,
    args=stage2_training_args,
    compute_metrics=compute_metrics,
    train_dataset=stage2_train_dataset,
    eval_dataset=stage2_eval_dataset,
    processing_class=processor.feature_extractor,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=5,
            early_stopping_threshold=0.001
        )
    ]
)

In [45]:
stage2_trainer.train()

Step,Training Loss,Validation Loss,Wer,Cer
50,0.243900,1.153414,0.866667,0.244192
100,0.218200,1.173240,0.863492,0.240534
150,0.208400,1.190817,0.863492,0.242729
200,0.191900,1.211196,0.866667,0.240717
250,0.195800,1.220210,0.866667,0.241815
300,0.188700,1.230686,0.866667,0.240534
350,0.165000,1.235309,0.863492,0.240534


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


TrainOutput(global_step=350, training_loss=0.20170205797467913, metrics={'train_runtime': 97.1107, 'train_samples_per_second': 81.433, 'train_steps_per_second': 20.389, 'total_flos': 2.0618457966662112e+17, 'train_loss': 0.20170205797467913, 'epoch': 2.121212121212121})

In [46]:
log_history_df = pd.DataFrame(stage2_trainer.state.log_history)

log_history_df.to_csv(
    f'{experiment_name}_stage2_log_history.csv',
    index=False,
    encoding='utf-8-sig'
)

best_metrics = get_best_dev_metrics(
    stage2_trainer,
    log_history_df
)

print(best_metrics)

{'best_dev_checkpoint': 'xls-r-300m-ckt-staged-bible-radio-to-chuklang-source-eval-stage2/checkpoint-100', 'best_dev_step': 100, 'best_dev_loss': np.float64(1.1732395887374878), 'best_dev_WER': np.float64(0.8634920634920635), 'best_dev_CER': np.float64(0.24053411377355038)}


In [47]:
processor.save_pretrained(stage2_training_args.output_dir)
stage2_trainer.save_model(stage2_training_args.output_dir)

stage2_trainer.push_to_hub()

processor.push_to_hub(
    model_repo_id,
    private=False,
    token=hf_token
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/tadgeis/xls-r-300m-ckt-staged-bible-radio-to-chuklang-source-eval-stage2/commit/4745e3c4856e5adc36b7922b6c89ad1098e147bb', commit_message='Upload processor', commit_description='', oid='4745e3c4856e5adc36b7922b6c89ad1098e147bb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/tadgeis/xls-r-300m-ckt-staged-bible-radio-to-chuklang-source-eval-stage2', endpoint='https://huggingface.co', repo_type='model', repo_id='tadgeis/xls-r-300m-ckt-staged-bible-radio-to-chuklang-source-eval-stage2'), pr_revision=None, pr_num=None)

In [ ]:
summary_rows = []
prediction_dfs = {}

for resource in resources_to_evaluate:
    row, results_df = evaluate_resource_test(
        resource=resource,
        experiment_name=experiment_name,
        model=stage2_trainer.model,
        batch_size=16
    )

    row['stage1_best_dev_checkpoint'] = stage1_best_metrics['best_dev_checkpoint']
    row['stage1_best_dev_step'] = stage1_best_metrics['best_dev_step']
    row['stage1_best_dev_loss'] = stage1_best_metrics['best_dev_loss']
    row['stage1_best_dev_WER'] = stage1_best_metrics['best_dev_WER']
    row['stage1_best_dev_CER'] = stage1_best_metrics['best_dev_CER']

    row['stage2_best_dev_checkpoint'] = best_metrics['best_dev_checkpoint']
    row['stage2_best_dev_step'] = best_metrics['best_dev_step']
    row['stage2_best_dev_loss'] = best_metrics['best_dev_loss']
    row['stage2_best_dev_WER'] = best_metrics['best_dev_WER']
    row['stage2_best_dev_CER'] = best_metrics['best_dev_CER']

    summary_rows.append(row)
    prediction_dfs[resource] = results_df

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    f'{experiment_name}_results_summary.csv',
    index=False,
    encoding='utf-8-sig'
)

summary_df

Filter:   0%|          | 0/429 [00:00<?, ? examples/s]

In [ ]:
files_to_download = [
    Path(f'{experiment_name}_stage1_log_history.csv'),
    Path(f'{experiment_name}_stage2_log_history.csv'),
    Path(f'{experiment_name}_results_summary.csv')
]

for resource in resources_to_evaluate:
    files_to_download.append(
        Path(f'{experiment_name}_{resource}_test_predictions.csv')
    )

for path in files_to_download:
    if path.exists():
        print(f'Ready to download: {path}')
    else:
        print(f'File not found: {path}')